# MCDD experiments

This notebook evaluates the detector configurations used in the MCDD study on
the HDF5 streams stored under `data/datasets/`.

Included methods:

- MCDD with sliding and growing windows;
- Traditional Single Hypothesis (TSH) baselines;
- River KSWIN;
- LORD under local dependence.

SEED is intentionally not included. The former *Comparison: Sliding vs Growing
Window* section is also omitted.


## Repository setup and imports

In [2]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repository_root(start: Path | None = None) -> Path:
    """Locate the repository root from the current working directory."""
    current = (start or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        if (candidate / "src" / "mcdd").is_dir():
            return candidate

    raise FileNotFoundError(
        "Repository root not found. Open this notebook from inside the repository."
    )


REPOSITORY_ROOT = find_repository_root()
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"

if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

from mcdd.experiments import (
    PAPER_CONFIGURATIONS,
    configuration_table,
    evaluate_single_stream,
    get_configuration,
    expected_dataset_paths,
    read_stream,
    run_archive_experiment,
    run_experiment_suite,
    summarize_archive_results,
    summarize_results,
)

DATA_DIRECTORY = REPOSITORY_ROOT / "data" / "datasets"
RESULTS_DIRECTORY = REPOSITORY_ROOT / "results"
PER_RUN_RESULTS = RESULTS_DIRECTORY / "per_run_results.csv"
SUMMARY_RESULTS = RESULTS_DIRECTORY / "summary_results.csv"

print(f"Repository root: {REPOSITORY_ROOT}")
print(f"Dataset directory: {DATA_DIRECTORY}")

Repository root: C:\Users\Fernando\Desktop\MCDD\multiple-contrast-drift-detection
Dataset directory: C:\Users\Fernando\Desktop\MCDD\multiple-contrast-drift-detection\data\datasets


## Experimental configurations

In [3]:
configurations = configuration_table(PAPER_CONFIGURATIONS)
display(
    configurations[
        [
            "name",
            "method",
            "window_mode",
            "window_size",
            "min_window_size",
            "max_window_size",
            "n_subwindows",
            "alpha",
            "correction",
            "description",
        ]
    ]
)

,name,method,window_mode,window_size,min_window_size,max_window_size,n_subwindows,alpha,correction,description
0,MCDD-S,mcdd,sliding,6000,NaN,NaN,10,0.01,fdr_by,"MCDD with a fixed 6,000-sample sliding window."
1,MCDD-G20k,mcdd,growing_dynamic,6000,6000.0,20000.0,10,0.01,fdr_by,"MCDD growing to 20,000 samples while keeping t..."
2,MCDD-G30k,mcdd,growing_dynamic,6000,6000.0,30000.0,10,0.01,fdr_by,"MCDD growing to 30,000 samples while keeping t..."
3,MCDD-G20kT,mcdd,growing_fixed,6000,6000.0,20000.0,10,0.01,fdr_by,"MCDD growing to 20,000 samples with fixed 600-..."
4,MCDD-G30kT,mcdd,growing_fixed,6000,6000.0,30000.0,10,0.01,fdr_by,"MCDD growing to 30,000 samples with fixed 600-..."
5,TSH-S,tsh,sliding,6000,NaN,NaN,10,0.01,None,"Traditional single KS test with a 6,000-sample..."
6,TSH-G20k,tsh,growing_dynamic,6000,6000.0,20000.0,10,0.01,None,Traditional single KS test with a window growi...
7,TSH-G30k,tsh,growing_dynamic,6000,6000.0,30000.0,10,0.01,None,Traditional single KS test with a window growi...
8,KSWIN,kswin,sliding,6000,NaN,NaN,10,0.01,None,"River KSWIN with window_size=6,000 and stat_si..."
9,LORD-LD,lord,sliding,6000,NaN,NaN,10,0.01,None,"LORD under local dependence using 6,000-sample..."


### Scoring convention

The notebook preserves the convention used in the original experiments:

- a valid detection must satisfy the strict comparison
  `start < detection < end`;
- for abrupt drift, the valid interval ends 2,000 observations after the true
  change point;
- if the first alarm lies outside the valid interval, it is recorded as a false
  alarm and the drift is not additionally counted as missed;
- if no alarm is raised, the drift is recorded as missed.

MCDD uses a latched alarm state. After a detection, the detector remains in the
detected state until `reset()` is called.


## Validate the HDF5 dataset files

In [4]:
dataset_paths = expected_dataset_paths(DATA_DIRECTORY)

pd.DataFrame(
    {
        "archive": [path.name for path in dataset_paths],
        "size_mb": [path.stat().st_size / (1024**2) for path in dataset_paths],
    }
)

,archive,size_mb
0,abrupt_normal.h5,461.243178
1,abrupt_exponential.h5,465.936771
2,abrupt_gamma.h5,458.806652
3,gradual_normal.h5,461.341128
4,gradual_exponential.h5,465.946480
5,gradual_gamma.h5,458.802858
6,incremental_normal.h5,461.278351
7,incremental_exponential.h5,465.946334
8,incremental_gamma.h5,458.851073


## Quick validation

This cell evaluates all ten configurations on the first stream in
`abrupt_normal.h5`. It is intended to verify imports, HDF5 reading, detector
construction, and result formatting before starting the full experiment.


In [5]:
quick_archive = DATA_DIRECTORY / "abrupt_normal.h5"
values, metadata = read_stream(quick_archive, row_index=0)

quick_results = [
    evaluate_single_stream(
        values,
        **metadata,
        configuration=configuration,
    )
    for configuration in PAPER_CONFIGURATIONS
]

quick_results_table = pd.DataFrame(quick_results)
display(
    quick_results_table[
        [
            "configuration",
            "alarm_index",
            "outcome",
            "drift_start",
            "valid_detection_end",
            "delay",
            "TP",
            "FP",
            "FN",
        ]
    ]
)

,configuration,alarm_index,outcome,drift_start,valid_detection_end,delay,TP,FP,FN
0,MCDD-S,40200,detected,40000,42000,200.0,1,0,0
1,MCDD-G20k,40200,detected,40000,42000,200.0,1,0,0
2,MCDD-G30k,40200,detected,40000,42000,200.0,1,0,0
3,MCDD-G20kT,40200,detected,40000,42000,200.0,1,0,0
4,MCDD-G30kT,40200,detected,40000,42000,200.0,1,0,0
5,TSH-S,40200,detected,40000,42000,200.0,1,0,0
6,TSH-G20k,40800,detected,40000,42000,800.0,1,0,0
7,TSH-G30k,40800,detected,40000,42000,800.0,1,0,0
8,KSWIN,6047,false_alarm,40000,42000,NaN,0,1,0
9,LORD-LD,40800,detected,40000,42000,800.0,1,0,0


## Run one detector on one dataset archive

Use this section to evaluate all 1,000 streams from one HDF5 archive with one
selected detector configuration.

For example:

- dataset: `abrupt_normal.h5`;
- detector: `MCDD-S`;
- executions: 1,000.

The section creates one per-run CSV containing 1,000 rows and one summary CSV
containing the aggregated metrics for that dataset/configuration pair.


In [8]:
RUN_SELECTED_EXPERIMENT = False

SELECTED_DATASET = "abrupt_normal.h5"
SELECTED_CONFIGURATION = "MCDD-S"

# Keep this as None to evaluate all 1,000 streams in the selected archive.
# Set it to a small integer, such as 2, for a quick execution test.
SELECTED_MAX_STREAMS = None

OVERWRITE_SELECTED_RESULTS = False

if RUN_SELECTED_EXPERIMENT:
    selected_archive = DATA_DIRECTORY / SELECTED_DATASET
    selected_configuration = get_configuration(
        SELECTED_CONFIGURATION
    )

    selected_results_directory = RESULTS_DIRECTORY / "selected"
    safe_configuration_name = (
        SELECTED_CONFIGURATION.lower().replace("-", "_")
    )
    result_stem = (
        f"{selected_archive.stem}__{safe_configuration_name}"
    )

    selected_per_run_file = (
        selected_results_directory
        / f"{result_stem}_per_run.csv"
    )
    selected_summary_file = (
        selected_results_directory
        / f"{result_stem}_summary.csv"
    )

    run_archive_experiment(
        archive_path=selected_archive,
        configuration=selected_configuration,
        output_file=selected_per_run_file,
        max_streams=SELECTED_MAX_STREAMS,
        overwrite=OVERWRITE_SELECTED_RESULTS,
        progress_every=25,
    )

    selected_summary = summarize_archive_results(
        per_run_file=selected_per_run_file,
        output_file=selected_summary_file,
    )

    display(selected_summary)

    print(f"Per-run results: {selected_per_run_file}")
    print(f"Summary results: {selected_summary_file}")
else:
    print(
        "Selected experiment is disabled. Set "
        "RUN_SELECTED_EXPERIMENT = True when ready."
    )

Selected experiment is disabled. Set RUN_SELECTED_EXPERIMENT = True when ready.


## Full experiment execution

The complete run evaluates:

- 9,000 streams;
- 10 detector configurations;
- 90,000 detector–stream combinations.

This can take a long time. Set `RUN_FULL_EXPERIMENTS = True` only when the nine
complete HDF5 archives are available and you are ready to start the full run.


In [ ]:
RUN_FULL_EXPERIMENTS = False
OVERWRITE_RESULTS = False

# Keep this as None for the complete paper experiment.
# Set it to a small integer, such as 2, for a reduced execution test.
MAX_STREAMS_PER_ARCHIVE = None

if RUN_FULL_EXPERIMENTS:
    run_experiment_suite(
        data_directory=DATA_DIRECTORY,
        output_file=PER_RUN_RESULTS,
        configurations=PAPER_CONFIGURATIONS,
        max_streams_per_archive=MAX_STREAMS_PER_ARCHIVE,
        overwrite=OVERWRITE_RESULTS,
        progress_every=25,
    )

    summary = summarize_results(
        per_run_file=PER_RUN_RESULTS,
        output_file=SUMMARY_RESULTS,
    )
    display(summary)
else:
    print(
        "Full execution is disabled. Set RUN_FULL_EXPERIMENTS = True "
        "and run this cell again when ready."
    )

## Inspect previously generated results

In [ ]:
if PER_RUN_RESULTS.is_file():
    per_run_results = pd.read_csv(PER_RUN_RESULTS)
    print(f"Per-run rows: {len(per_run_results):,}")
    display(per_run_results.head())

    summary = summarize_results(
        per_run_file=PER_RUN_RESULTS,
        output_file=SUMMARY_RESULTS,
    )
    display(summary)
else:
    print(f"No per-run result file found at {PER_RUN_RESULTS}.")